Log

In [1]:
%run Utils_Log

StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

In [ ]:
setup_log("Silver_mares")

### Librerias necesarias :

In [1]:
#%pip uninstall numpy -y
#%pip install "numpy<2.0.0" xarray netCDF4 pandas dask h5py --quiet

StatementMeta(, 299534d1-9c68-4d7b-9e99-099282f2191d, 8, Finished, Available, Finished, True)

Found existing installation: numpy 1.26.4
Not uninstalling numpy at /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages, outside environment /nfs4/pyenv-e3853195-3025-444d-9117-90418514dd80
Can't uninstall 'numpy'. No files were found to uninstall.
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow-skinny 2.12.2 requires packaging<25, but you have packaging 26.2 which is incompatible.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.
ds-copilot 0.1.25.2.28 requires pandas<3.0.0,>=1.5.0, but you have pandas 3.0.3 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



Logs

In [2]:
from pathlib import Path
import xarray as xr
import pandas as pd
from pyspark.sql.functions import col, to_date, round, sqrt, current_timestamp, when, lit, current_timestamp


BANCOS_PESCA = [
    {"nombre": "Zona I - Vigo",          "latitud": 42.22356, "longitud": -8.82484},
    {"nombre": "Zona II - Pontevedra",   "latitud": 42.36464, "longitud": -8.81874},
    {"nombre": "Zona III - Arousa",      "latitud": 42.50921, "longitud": -8.94458},
    {"nombre": "Zona IV - Muros",        "latitud": 42.69498, "longitud": -9.08367},
    {"nombre": "Zona V - Fisterra",      "latitud": 42.85957, "longitud": -9.21355},
    {"nombre": "Zona VI - Costa da Morte", "latitud": 43.229,   "longitud": -9.05312},
    {"nombre": "Zona VII - Coruña-Ferrol", "latitud": 43.4249,  "longitud": -8.3557},
    {"nombre": "Zona VIII - Cedeira",    "latitud": 43.76145, "longitud": -8.00771},
    {"nombre": "Zona IX - Mariña",       "latitud": 43.71027, "longitud": -7.24132},
]

BRONZE_DIR = Path("/lakehouse/default/Files/Bronze/Oceanografia")
VARIABLES = ["so", "thetao", "uo", "vo"]

fragmentos = []

for banco in BANCOS_PESCA:
    nombre = banco["nombre"]
    nc_path = BRONZE_DIR / nombre / f"{nombre}.nc"
    
    if nc_path.exists():
        df_temp = xr.open_dataset(nc_path)[VARIABLES].squeeze().to_dataframe().reset_index()
        df_temp.insert(0, "zona", nombre)
        fragmentos.append(df_temp)

df_pandas = pd.concat(fragmentos, ignore_index=True)
df_pandas.columns = [c.lower().replace(" ", "_") for c in df_pandas.columns]

StatementMeta(, 299534d1-9c68-4d7b-9e99-099282f2191d, 11, Finished, Available, Finished, False)

In [3]:
df_bronze_spark = spark.createDataFrame(df_pandas)

# Renombre y tipado
df_silver = df_bronze_spark \
    .withColumnRenamed("time", "data") \
    .withColumnRenamed("thetao", "temperatura_c") \
    .withColumnRenamed("so", "salinidad_psu") \
    .withColumnRenamed("uo", "corriente_u") \
    .withColumnRenamed("vo", "corriente_v")

# Casteamos la fecha a DateType estricto para que coincida con ventas_silver
# Redondeamos a 3 decimales 
df_silver = df_silver \
    .withColumn("data", to_date(col("data"))) \
    .withColumn("temperatura_c", round(col("temperatura_c"), 3)) \
    .withColumn("salinidad_psu", round(col("salinidad_psu"), 3)) \
    .withColumn("corriente_u", round(col("corriente_u"), 3)) \
    .withColumn("corriente_v", round(col("corriente_v"), 3)) \
    .withColumn("velocidad_mar", round(sqrt(pow(col("corriente_u"), 2) + pow(col("corriente_v"), 2)), 3)) \
    .withColumn("fecha_carga", current_timestamp())

log("Capa Silver generada: Temperatura, salinidad y corrientes estandarizadas.")
print("Capa Silver generada: Temperatura y salinidad estandarizadas.")

StatementMeta(, 299534d1-9c68-4d7b-9e99-099282f2191d, 12, Finished, Available, Finished, False)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Cannot convert pyarrow.lib.ChunkedArray to pyarrow.lib.Array
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.


Capa Silver generada: Temperatura y salinidad estandarizadas.


In [6]:
#df_silver.head()

StatementMeta(, 86de97ef-91ba-4671-9906-4490588608c6, 15, Finished, Available, Finished, False)

Row(zona='Zona I - Vigo', data=datetime.date(2015, 1, 1), salinidad_psu=34.931, temperatura_c=12.026, corriente_u=0.013, corriente_v=0.111, depth=0.49402499198913574, latitude=42.25, longitude=-8.833333015441895, velocidad_mar=0.112, fecha_carga=datetime.datetime(2026, 6, 4, 17, 33, 25, 522835))

### Calidad del Dato

In [4]:
df_calidad = (
    df_silver
    .withColumn(
        "estado_calidad",
        when(col("data").isNull(), "REJECT")
        .when(col("temperatura_c").isNull(), "REJECT")
        .when(col("salinidad_psu").isNull(), "REJECT")
        .when(col("temperatura_c") < -5, "REJECT")
        .when(col("temperatura_c") > 40, "REJECT")
        .when(col("salinidad_psu") < 0, "REJECT")
        .when(col("data") > lit("2025-12-31"), "QUARANTINE")
        .when((col("temperatura_c") < 5) | (col("temperatura_c") > 30), "QUARANTINE")
        .when((col("salinidad_psu") < 20) | (col("salinidad_psu") > 45), "QUARANTINE")
        .when(col("velocidad_mar") > 5, "QUARANTINE")
        .otherwise("VALIDO")
    )
)

StatementMeta(, 299534d1-9c68-4d7b-9e99-099282f2191d, 13, Finished, Available, Finished, False)

In [5]:
df_silver = df_calidad.filter(col("estado_calidad") == "VALIDO")
df_quarantine = df_calidad.filter(col("estado_calidad") == "QUARANTINE")
df_reject = df_calidad.filter(col("estado_calidad") == "REJECT")

StatementMeta(, 299534d1-9c68-4d7b-9e99-099282f2191d, 14, Finished, Available, Finished, False)

## Limpieza final de las columnas

In [6]:
for c in ["depth", "corriente_u", "corriente_v", "estado_calidad"]:
    if c in df_silver.columns:
        df_silver = df_silver.drop(c)
    if c in df_quarantine.columns:
        df_quarantine = df_quarantine.drop(c)
    if c in df_reject.columns:
        df_reject = df_reject.drop(c)

StatementMeta(, 299534d1-9c68-4d7b-9e99-099282f2191d, 15, Finished, Available, Finished, False)

In [7]:
print(f"Total      : {df_calidad.count()}")
print(f"Silver     : {df_silver.count()}")
print(f"Quarantine : {df_quarantine.count()}")
print(f"Reject     : {df_reject.count()}")

StatementMeta(, 299534d1-9c68-4d7b-9e99-099282f2191d, 16, Finished, Available, Finished, False)

Total      : 37224
Silver     : 36162
Quarantine : 1062
Reject     : 0


In [8]:
# 4. CARGA EN SILVER
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("oceanografia_silver")


df_quarantine \
    .withColumn("timestamp_revision", current_timestamp()) \
    .write.format("delta").mode("append").saveAsTable("silver_quarantine")


df_reject \
    .withColumn("timestamp_rechazo", current_timestamp()) \
    .write.format("delta").mode("append").saveAsTable("silver_rejected")

StatementMeta(, 299534d1-9c68-4d7b-9e99-099282f2191d, 17, Finished, Available, Finished, False)

In [9]:
from notebookutils import mssparkutils

# Esto sí apaga el clúster físico desde el código
#mssparkutils.session.stop()

StatementMeta(, 299534d1-9c68-4d7b-9e99-099282f2191d, 18, Finished, Available, Finished, False)